In [1]:
# reference example
from nequip.data import dataset_from_config
from nequip.utils import Config
#from nequip.utils.misc import get_default_device_name
#from nequip.utils.config import _GLOBAL_ALL_ASKED_FOR_KEYS

from nequip.model import model_from_config


default_config = dict(
    root="./",
    tensorboard=False,
    wandb=False,
    model_builders=[
        "SimpleIrrepsConfig",
        "EnergyModel",
        "PerSpeciesRescale",
        "StressForceOutput",
        "RescaleEnergyEtc",
    ],
    dataset_statistics_stride=1,
    device='cuda',
    default_dtype="float64",
    model_dtype="float32",
    allow_tf32=True,
    verbose="INFO",
    model_debug_mode=False,
    equivariance_test=False,
    grad_anomaly_mode=False,
    gpu_oom_offload=False,
    append=False,
    warn_unused=False,
    _jit_bailout_depth=2,  # avoid 20 iters of pain, see https://github.com/pytorch/pytorch/issues/52286
    # Quote from eelison in PyTorch slack:
    # https://pytorch.slack.com/archives/CDZD1FANA/p1644259272007529?thread_ts=1644064449.039479&cid=CDZD1FANA
    # > Right now the default behavior is to specialize twice on static shapes and then on dynamic shapes.
    # > To reduce warmup time you can do something like setFusionStrartegy({{FusionBehavior::DYNAMIC, 3}})
    # > ... Although we would wouldn't really expect to recompile a dynamic shape fusion in a model,
    # > provided broadcasting patterns remain fixed
    # We default to DYNAMIC alone because the number of edges is always dynamic,
    # even if the number of atoms is fixed:
    _jit_fusion_strategy=[("DYNAMIC", 3)],
    # Due to what appear to be ongoing bugs with nvFuser, we default to NNC (fuser1) for now:
    # TODO: still default to NNC on CPU regardless even if change this for GPU
    # TODO: default for ROCm?
    _jit_fuser="fuser1",
)

# All default_config keys are valid / requested
#_GLOBAL_ALL_ASKED_FOR_KEYS.update(default_config.keys())

In [2]:
config = Config.from_file('./configs/example_SpinGNNPlus.yaml', defaults=default_config)
    

dataset = dataset_from_config(config, prefix="dataset")

validation_dataset = None

dataset[0]

AtomicData(atom_types=[21, 1], cell=[3, 3], edge_cell_shift=[364, 3], edge_index=[2, 364], forces=[21, 3], pbc=[3], pos=[21, 3], total_energy=[1])

In [3]:
config

{'_jit_bailout_depth': 2, '_jit_fusion_strategy': [('DYNAMIC', 3)], '_jit_fuser': 'fuser1', 'root': 'results/aspirin', 'tensorboard': False, 'wandb': False, 'model_builders': ['allegro.model.SpinGNNPlus', 'PerSpeciesRescale', 'ParaStressForceSpinForceOutput', 'RescaleEnergyEtc'], 'dataset_statistics_stride': 1, 'device': 'cuda', 'default_dtype': 'float32', 'model_dtype': 'float32', 'allow_tf32': True, 'verbose': 'debug', 'model_debug_mode': False, 'equivariance_test': False, 'grad_anomaly_mode': False, 'gpu_oom_offload': False, 'append': True, 'warn_unused': False, 'run_name': 'example', 'seed': 123456, 'dataset_seed': 123456, 'r_max': 6.0, 'avg_num_neighbors': 'auto', 'BesselBasis_trainable': True, 'PolynomialCutoff_p': 6, 'l_max': 2, 'parity': 'o3_full', 'num_layers': 2, 'env_embed_multiplicity': 64, 'embed_initial_edge': True, 'two_body_latent_mlp_latent_dimensions': [128, 256, 512, 1024], 'two_body_latent_mlp_nonlinearity': 'silu', 'two_body_latent_mlp_initialization': 'uniform', '

In [4]:
# Trainer
from nequip.train.trainer import Trainer
from e3nn import o3

trainer = Trainer(model=None, **Config.as_dict(config))

# what is this
# to update wandb data?
config.update(trainer.params)

# = Train/test split =
trainer.set_dataset(dataset, validation_dataset)

config['model_input_fields'] = {'magmoms': o3.Irreps('1x1e')}

# = Build model =
final_model = model_from_config(
    config=config, initialize=True, dataset=trainer.dataset_train
)

DEBUG:root:* Initialize Output
  ...generate file name results/aspirin/example/log
  ...open log file results/aspirin/example/log
  ...generate file name results/aspirin/example/metrics_epoch.csv
  ...open log file results/aspirin/example/metrics_epoch.csv
  ...generate file name results/aspirin/example/metrics_initialization.csv
  ...open log file results/aspirin/example/metrics_initialization.csv
  ...generate file name results/aspirin/example/metrics_batch_train.csv
  ...open log file results/aspirin/example/metrics_batch_train.csv
  ...generate file name results/aspirin/example/metrics_batch_val.csv
  ...open log file results/aspirin/example/metrics_batch_val.csv
  ...generate file name results/aspirin/example/best_model.pth
  ...generate file name results/aspirin/example/last_model.pth
  ...generate file name results/aspirin/example/trainer.pth
  ...generate file name results/aspirin/example/config.yaml
Torch device: cuda
instantiate Loss
...Loss_param = dict(
...   optional_args 

Is ij diagonal
tensor(False)


/pscratch/sd/v/vladygin/doped-Si_project/MLFF_TDEP/testbench/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(
/pscratch/sd/v/vladygin/doped-Si_project/MLFF_TDEP/testbench/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(
/pscratch/sd/v/vladygin/doped-Si_project/MLFF_TDEP/testbench/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation

Is ij diagonal
False


/pscratch/sd/v/vladygin/doped-Si_project/MLFF_TDEP/testbench/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(
/pscratch/sd/v/vladygin/doped-Si_project/MLFF_TDEP/testbench/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(
/pscratch/sd/v/vladygin/doped-Si_project/MLFF_TDEP/testbench/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation

Is ij diagonal
tensor(False)


/pscratch/sd/v/vladygin/doped-Si_project/MLFF_TDEP/testbench/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(
/pscratch/sd/v/vladygin/doped-Si_project/MLFF_TDEP/testbench/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(
/pscratch/sd/v/vladygin/doped-Si_project/MLFF_TDEP/testbench/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation

Is ij diagonal
tensor(True)


/pscratch/sd/v/vladygin/doped-Si_project/MLFF_TDEP/testbench/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(
/pscratch/sd/v/vladygin/doped-Si_project/MLFF_TDEP/testbench/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(
instantiate ScalarMLP
   optional_args :                                mlp_output_dimension
   optional_args :                                               field
   optional_args :                                           out_field
   optional_args :                         

instantiate ScalarMLP
   optional_args :                                               field
   optional_args :                                           out_field
   optional_args :                                mlp_output_dimension
   optional_args :                               mlp_latent_dimensions
...ScalarMLP_param = dict(
...   optional_args = {'mlp_nonlinearity': 'silu', 'mlp_initialization': 'uniform', 'mlp_dropout_p': 0.0, 'mlp_batchnorm': False, 'field': 'edge_features', 'out_field': 'edge_energy_TENN', 'mlp_latent_dimensions': [], 'mlp_output_dimension': 1},
...   positional_args = {'irreps_in': {'pos': 1x1oe, 'edge_index': None, 'node_attrs': 3x0ee, 'node_features': 3x0ee, 'edge_embedding': 8x0ee, 'edge_cutoff': 1x0ee, 'edge_attrs': 3x0ee+3x1eo+3x2ee, 'edge_features': 1024x0ee, 'edge_features_MSENN_J': 1x0ee+1x1ee+1x2ee, 'edge_features_MSENN_A': 1x0ee+1x2ee, 'edge_energy': 1x0ee, 'edge_K': 1x0ee, 'atomic_energy': 1x0ee, 'edge_spin_distance_embdedding': 8x0ee, 'node_spin_

In [5]:

import torch
from torch.nn.functional import one_hot
from nequip.data import AtomicData, AtomicDataDict
from torch.nn.functional import one_hot
from e3nn.nn import FullyConnectedNet
from allegro import with_edge_spin_length
from allegro import _keys
from torch import nn
import math

data0 = AtomicData.to_AtomicDataDict(dataset[0])
data0[AtomicDataDict.SPIN_KEY] = torch.randn_like(data0['pos'], requires_grad=True)



In [6]:
final_model

GraphModel(
  (model): RescaleOutput(
    (model): ParaStressSpinForceOutput(
      (func): SequentialGraphNetwork(
        (one_hot): OneHotAtomEncoding()
        (radial_basis): RadialBasisEdgeEncoding(
          (basis): NormalizedBasis(
            (basis): BesselBasis()
          )
          (cutoff): PolynomialCutoff()
        )
        (spharm): SphericalHarmonicEdgeAttrs(
          (sh): SphericalHarmonics()
        )
        (allegro_MSENN): Allegro_Module_MSENN(
          (latents): ModuleList(
            (0-1): 2 x ScalarMLPFunction(
              (_forward): RecursiveScriptModule(original_name=GraphModule)
            )
          )
          (env_embed_mlps): ModuleList(
            (0-1): 2 x ScalarMLPFunction(
              (_forward): RecursiveScriptModule(original_name=GraphModule)
            )
          )
          (tps): ModuleList(
            (0-1): 2 x RecursiveScriptModule(original_name=GraphModule)
          )
          (linears): ModuleList(
            (0-1):

In [7]:
from e3nn import o3

trainer.model = final_model

In [8]:
576/3/3

64.0

In [9]:
576/9.

64.0

In [10]:
import torch
from torch.nn.functional import one_hot
from nequip.data import AtomicData, AtomicDataDict
from torch.nn.functional import one_hot
from e3nn.nn import FullyConnectedNet
    
from torch import nn
import math

data_new = final_model(data0)

### Checking cartesian

In [13]:
import torch

from e3nn.io import CartesianTensor
from e3nn.util.jit import compile_mode


@compile_mode("script")
class Cartesian_nn(torch.nn.Module):
    def __init__(self, formula):
        super().__init__()
        ct = CartesianTensor(formula)
        rtp = ct.reduced_tensor_products()
        self.Q = rtp.change_of_basis
        self.last_dim = 3 ** len(self.indices)
        self.indices = ct.indices

    def forward(self, data: torch.Tensor):
        data = (data @ self.Q.flatten(-len(self.indices))).view(-1, self.last_dim)
        return data

In [18]:
output_nonscalar_irreps_J = CartesianTensor("ij=ij")
output_nonscalar_irreps_ani = CartesianTensor("ij=ji")

nonscalr_transform_last_Q_J = output_nonscalar_irreps_J.reduced_tensor_products().change_of_basis.flatten(
    -len(output_nonscalar_irreps_J.indices) 
)

nonscalr_transform_last_Q_ani = output_nonscalar_irreps_ani.reduced_tensor_products().change_of_basis.flatten(
        -len(output_nonscalar_irreps_J.indices)
)

/pscratch/sd/v/vladygin/doped-Si_project/MLFF_TDEP/testbench/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(
/pscratch/sd/v/vladygin/doped-Si_project/MLFF_TDEP/testbench/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(
/pscratch/sd/v/vladygin/doped-Si_project/MLFF_TDEP/testbench/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation

In [19]:
nonscalr_transform_last_Q_J

tensor([[ 0.5774,  0.0000,  0.0000,  0.0000,  0.5774,  0.0000,  0.0000,  0.0000,
          0.5774],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.7071,  0.0000, -0.7071,
          0.0000],
        [ 0.0000,  0.0000, -0.7071,  0.0000,  0.0000,  0.0000,  0.7071,  0.0000,
          0.0000],
        [ 0.0000,  0.7071,  0.0000, -0.7071,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000],
        [ 0.0000,  0.0000,  0.7071,  0.0000,  0.0000,  0.0000,  0.7071,  0.0000,
          0.0000],
        [ 0.0000,  0.7071,  0.0000,  0.7071,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000],
        [-0.4082,  0.0000,  0.0000,  0.0000,  0.8165,  0.0000,  0.0000,  0.0000,
         -0.4082],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.7071,  0.0000,  0.7071,
          0.0000],
        [-0.7071,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.7071]])

In [20]:
output_nonscalar_irreps_J

1x0ee+1x1ee+1x2ee

In [21]:
output_nonscalar_irreps_ani

1x0ee+1x2ee

In [22]:
final_model.irreps_out

{'pos': 1x1oe,
 'edge_index': None,
 'edge_cell_shift': None,
 'cell': 1x1oe,
 'batch': None,
 'ptr': None,
 'atom_types': None,
 'magmoms': 1x0ee,
 'node_attrs': 3x0ee,
 'node_features': 3x0ee,
 'edge_embedding': 8x0ee,
 'edge_cutoff': 1x0ee,
 'edge_attrs': 3x0ee+3x1eo+3x2ee,
 'edge_features': 1024x0ee,
 'edge_features_MSENN_J': 1x0ee+1x1ee+1x2ee,
 'edge_features_MSENN_A': 1x0ee+1x2ee,
 'edge_energy': 1x0ee,
 'edge_K': 1x0ee,
 'atomic_energy': 1x0ee,
 'edge_spin_distance_embdedding': 8x0ee,
 'node_spin_length': 1x0ee,
 'edge_energy_BQ': 1x0ee,
 'atomic_energy_BQ': 1x0ee,
 'edge_energy_J': 1x0ee,
 'atomic_energy_J': 1x0ee,
 'atomic_energy_A': 1x0ee,
 'edge_energy_TENN': 1x0ee,
 'edge_spin': 1x0ee,
 'atomic_energy_TENN': 1x0ee,
 'total_energy': 1x0ee,
 'forces': 1x1oe,
 'spin_forces': 1x1oo,
 'stress': 1x1oe,
 'virial': 1x1oe,
 'atom_virial': 1x1oe}

In [29]:
from allegro.nn._strided import Linear
from e3nn.nn import NormActivation

tps_irreps_out = final_model.irreps_out['edge_features_MSENN_J']
pad_to_alignment = 1


final_nonscalar_J = torch.nn.Sequential()
final_nonscalar_J.append(
    NormActivation(
        irreps_in=[(1, ir) for _, ir in tps_irreps_out],
        # norm is an even scalar, so we use silu
        scalar_nonlinearity=torch.nn.functional.silu,
        normalize=True,
        epsilon=1e-8,
        bias=False,
    )
)
final_nonscalar_J.append(
    Linear(
        [(1, ir) for _, ir in tps_irreps_out],
        output_nonscalar_irreps_J,
        shared_weights=True,
        internal_weights=True,
        pad_to_alignment=pad_to_alignment,
    )
)

final_nonscalar_ani = torch.nn.Sequential()
final_nonscalar_ani.append(
    Linear(
        [(1, ir) for _, ir in tps_irreps_out],
        [(1, ir) for _, ir in tps_irreps_out],
        shared_weights=True,
        internal_weights=True,
        pad_to_alignment=pad_to_alignment,
    )
)
final_nonscalar_ani.append(
    NormActivation(
        irreps_in=[(1, ir) for _, ir in tps_irreps_out],
        # norm is an even scalar, so we use silu
        scalar_nonlinearity=torch.nn.functional.silu,
        normalize=True,
        epsilon=1e-8,
        bias=False,
    )
)
final_nonscalar_ani.append(
    Linear(
        [(1, ir) for _, ir in tps_irreps_out],
        output_nonscalar_irreps_ani,
        shared_weights=True,
        internal_weights=True,
        pad_to_alignment=pad_to_alignment,
    )
)

/pscratch/sd/v/vladygin/doped-Si_project/MLFF_TDEP/testbench/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(
/pscratch/sd/v/vladygin/doped-Si_project/MLFF_TDEP/testbench/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(
/pscratch/sd/v/vladygin/doped-Si_project/MLFF_TDEP/testbench/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation

Sequential(
  (0): RecursiveScriptModule(original_name=GraphModule)
  (1): NormActivation(
    (norm): Norm(1x0ee+1x1ee+1x2ee)
    (scalar_multiplier): ElementwiseTensorProduct(3x0ee x 1x0ee+1x1ee+1x2ee -> 1x0ee+1x1ee+1x2ee | 3 paths | 0 weights)
  )
  (2): RecursiveScriptModule(original_name=GraphModule)
)

In [34]:
features = data_new['edge_features_MSENN_J']

features_J = (
    final_nonscalar_J(features) @ nonscalr_transform_last_Q_J
)#.view(-1, 9)
features_ani = (
    final_nonscalar_ani(features) @ nonscalr_transform_last_Q_ani
)#.view(-1, 9)

In [36]:
features.shape, features_J.shape

(torch.Size([364, 9]), torch.Size([364, 1, 9]))

In [37]:
nonscalr_transform_last_Q_J.shape

torch.Size([9, 9])

In [38]:
nonscalr_transform_last_Q_ani.shape

torch.Size([6, 9])